In [41]:
import time
import cv2 as cv
from ultralytics import YOLO

In [25]:
model = YOLO('yolo11n-seg.pt')

In [4]:
from ultralytics import YOLO

# Load an official or custom model
model = YOLO("yolo11n.pt")  # Load an official Detect model
model = YOLO("yolo11n-seg.pt")  # Load an official Segment model
model = YOLO("yolo11n-pose.pt")  # Load an official Pose model


In [36]:
cap = cv.VideoCapture(0)

In [27]:
def fps(start, end):
    return int(1//(end-start))

In [2]:
%%writefile reid_config.yaml
tracker_type: 'bytetrack'
with_reid: True
reid_model: 'auto'


Overwriting reid_config.yaml


In [37]:
try:
    while cap.isOpened():
        ret, image = cap.read()
        if not ret:
            print('No camera detected, aborting')
            break
        start = time.perf_counter()
        results = model.track(source=image,tracker="reid_config.yaml",persist=True,verbose=False)
        end = time.perf_counter()
        segments = results[0].plot()
        """cv.putText(segments, f'FPS: {fps(start, end)}', (10, 30),
                   cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)"""
        
        cv.imshow('Image Segmentation', segments)
        key = cv.waitKey(1)
        if key & 0xFF == ord('q'):
            print('Exit sequence initiated')
            break

finally:
    cap.release()
    cv.destroyAllWindows()

AssertionError: Only 'bytetrack' and 'botsort' are supported for now, but got 'bytetrack.yaml'

In [43]:
import cv2
from ultralytics import YOLO

# 1. Load the three distinct models
model_det = YOLO('yolov8n.pt')      # Detection
model_seg = YOLO('yolov8n-seg.pt') # Segmentation
model_pose = YOLO('yolov8n-pose.pt') # Pose Estimation

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

# Note: Running three models sequentially will significantly increase latency
print("Starting camera feed. Combining Detection, Segmentation, and Pose tracking.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # We use 'frame' as the base image, then overwrite/plot on it sequentially
    
    # A. Run Detection Tracking FIRST (to establish primary bounding boxes)
    results_det = model_det.track(
        source=frame, 
        tracker="reid_config.yaml", 
        persist=True, 
        verbose=False,
        conf=0.5
    )
    # Plot detection results onto the frame
    annotated_frame = results_det.plot() 
    
    # B. Run Segmentation Tracking SECOND (using the already annotated frame as source image)
    results_seg = model_seg.track(
        source=annotated_frame, # Input is the image output from the previous step
        tracker="reid_config.yaml", 
        persist=True, 
        verbose=False,
        conf=0.5
    )
    # Plot segmentation results (masks + boxes) onto the frame
    annotated_frame = results_seg.plot()

    # C. Run Pose Estimation Tracking LAST
    results_pose = model_pose.track(
        source=annotated_frame, # Input is the image output from the previous step
        tracker="reid_config.yaml", 
        persist=True, 
        verbose=False,
        conf=0.5
    )
    # Plot pose estimation results (keypoints + boxes) onto the final frame
    final_display_frame = results_pose.plot()
    

    # Display the final combined frame
    cv2.imshow("Combined Multi-Model Tracking", final_display_frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Starting camera feed. Combining Detection, Segmentation, and Pose tracking.


AttributeError: 
            'IterableSimpleNamespace' object has no attribute 'track_buffer'. This may be caused by a modified or out of date ultralytics
            'default.yaml' file.
Please update your code with 'pip install -U ultralytics' and if necessary replace
            C:\WorkSpace\Enhanced_Activity_monitor\.venv\Lib\site-packages\ultralytics\cfg\default.yaml with the latest version from
            https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/default.yaml
            

In [1]:
pip install -U ultralytics

In [51]:
import cv2
from ultralytics import YOLO
import collections # Python's default dictionary is fine, but collections is robust
import tracker import Tracker

# Load the detection model
model = YOLO('yolov8n.pt') 

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

# Dictionary to store the history of tracked points for visualization
# Format: {track_id: deque([center_point1, center_point2, ...])}
track_history = collections.defaultdict(lambda: collections.deque(maxlen=30))

print("Tracking only 'person' class (ID 0). Monitoring activity.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # Run tracking for ONLY class 0 (person)
    results = model.track(
        source=frame, 
        tracker="reid_config.yaml", 
        persist=True, 
        verbose=False,
        conf=0.5,
        classes=[0] # <-- This ensures ONLY people are detected/tracked
    )

    # Get the boxes and track IDs
    boxes = results[0].boxes.xywh.cpu() # Use xywh for easy centroid calculation
    track_ids = results[0].boxes.id.int().cpu().tolist() if results[0].boxes.id is not None else []

    # Iterate over the detections and plot their historical paths (activity tracking)
    for box, track_id in zip(boxes, track_ids):
        x, y, w, h = box
        # Calculate the center point (centroid) of the bounding box
        center = (int(x), int(y)) 
        
        # Store the history of this person's movement
        track_history[track_id].append(center)

        # Visualize the path (basic activity monitoring)
        points = track_history[track_id]
        for i in range(1, len(points)):
            # Draw lines connecting previous 30 frames worth of movement
            cv2.line(frame, points[i-1], points[i], (0, 255, 0), 2)

    # Display the resulting frame (we skip results.plot() to use our manual line drawing)
    cv2.imshow("Person Activity Monitoring", frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


SyntaxError: invalid syntax (682619884.py, line 4)

In [4]:
!pip install -U --force-reinstall ultralytics

  Using cached ultralytics-8.3.235-py3-none-any.whl.metadata (37 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached matplotlib-3.10.7-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.0.0-cp310-cp310-win_amd64.whl.metadata (9.0 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl.metadata (2.4 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached torch-2.9.1-cp310-cp310-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.24.1-cp310-cp310-win_amd64.whl.metadata (5.9 kB)
  Using cached psutil-7.1.3-cp37-abi3-win_amd64.whl.metadata (23 kB)
  Using cached polars-1.35.2-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.1 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.


In [5]:
%%writefile reid_config1.yaml
tracker_type: 'botsort'
with_reid: True
reid_model: 'auto'


Writing reid_config1.yaml


In [7]:
import yaml
import os

file_path = "reid_config1.yaml"

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        config_data = yaml.safe_load(f)
        print(f"Contents of {file_path}:\n{config_data}")
else:
    print(f"Error: {file_path} not found.")



Contents of reid_config1.yaml:
{'tracker_type': 'botsort', 'with_reid': True, 'reid_model': 'auto'}


In [6]:
import cv2
from ultralytics import YOLO

# Load a pose estimation model
model = YOLO('yolov8n-pose.pt') 

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

print("Starting camera feed with Pose Estimation and Tracking. Press 'q' to quit.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # Run tracking for ONLY class 0 (person)
    # The pose model will detect people AND their keypoints
    results = model.track(source=frame, tracker="reid_config1.yaml", persist=True, verbose=False,conf=0.5,classes=0 )# Explicitly filter to only track 'person' class)

    # results.plot() automatically draws bounding boxes, track IDs, 
    # and the keypoint skeletons for posture visualization.
    annotated_frame = results.plot() 

    # If you want to access the raw keypoint data for analysis:
    # if results[0].keypoints is not None and results[0].boxes.id is not None:
    #     keypoints = results[0].keypoints.xy.cpu().numpy()
    #     track_ids = results[0].boxes.id.cpu().numpy()
    #     for kpts, track_id in zip(keypoints, track_ids):
    #         # kpts is an array of [x, y] for ~17 joints for that specific track_id
    #         print(f"Track ID {track_id} Nose position: {kpts[0]}")


    # Display the resulting frame
    cv2.imshow("Person Pose & Tracking with ReID", annotated_frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Starting camera feed with Pose Estimation and Tracking. Press 'q' to quit.


AttributeError: 
            'IterableSimpleNamespace' object has no attribute 'model'. This may be caused by a modified or out of date ultralytics
            'default.yaml' file.
Please update your code with 'pip install -U ultralytics' and if necessary replace
            C:\WorkSpace\Enhanced_Activity_monitor\.venv\Lib\site-packages\ultralytics\cfg\default.yaml with the latest version from
            https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/default.yaml
            

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("atalaydenknalbant/Kinetics-700")

In [ ]:
from ultralytics import YOLO

# Load the model and run the tracker with a custom configuration file
model = YOLO("yolo11n.pt")
results = model.track(source="https://youtu.be/LNwODJXcvt4", tracker="custom_tracker.yaml")

In [ ]:
wget https://people.eecs.berkeley.edu/~kanazawa/cachedir/hmr/models.tar.gz && tar -xf models.tar.gz